# Wind Data Sources

HyPlan supports multiple wind data sources for mission planning:

| Source | Resolution | Coverage | Auth | Best for |
|--------|-----------|----------|------|----------|
| **MERRA-2** | 0.5° × 0.625°, 3-hourly | 1980–present | EarthData | Historical planning |
| **GMAO GEOS-FP** | 0.25°, 3-hourly | Last ~30 days | None | Near-real-time |
| **GFS** | 0.25°, 3-hourly | 0–16 day forecast | None | Operational forecasts |
| **Constant** | Uniform | N/A | None | Sensitivity analysis |
| **Still air** | Zero wind | N/A | None | Baseline |

This notebook demonstrates each source. For how wind **affects**
flight planning (crab angles, Dubins paths, mission timing), see
`wind_effects.ipynb`.

In [12]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import datetime
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import folium

from hyplan import (
    FlightLine, DubinsPath3D,
    KingAirB200,
    Airport, initialize_data,
    compute_flight_plan, plot_flight_plan, plot_altitude_trajectory,
    ureg,
)
from hyplan.aircraft import TwinOtter, NASA_ER2
from hyplan.instruments import GLiHT_VNIR, AVIRISClassic
from hyplan.winds import (
    StillAirField, ConstantWindField,
    wind_field_from_plan,
)
from hyplan.swath import generate_swath_polygon, calculate_swath_widths
from hyplan.flight_box import box_around_center_line


## 1. Setup: Flight Lines and Aircraft

We set up a simple mission with three east–west flight lines near Santa
Barbara at FL200 (20,000 ft). The B200 cruises at ~250 kt TAS at this
altitude, so wind effects on timing will be significant.

In [13]:
initialize_data(countries=["US"])

aircraft = KingAirB200()
departure = Airport("KSBA")

# Three east-west flight lines
flight_lines = []
for i, lat in enumerate([34.40, 34.35, 34.30]):
    az = 90.0 if i % 2 == 0 else 270.0  # alternating direction
    fl = FlightLine.center_length_azimuth(
        lat=lat, lon=-119.8,
        length=ureg.Quantity(80, "km"),
        az=az,
        altitude_msl=ureg.Quantity(20000, "feet"),
        site_name=f"Coastal_L{i+1:02d}",
    )
    flight_lines.append(fl)

print(f"Aircraft: {type(aircraft).__name__}")
print(f"Departure: {departure.name} ({departure.icao_code})")
print(f"\n{len(flight_lines)} flight lines (alternating E/W):")
for fl in flight_lines:
    print(f"  {fl.site_name}: {fl.length.to(ureg.km):.1f}, az={fl.az12:.0f}")

Aircraft: KingAirB200
Departure: Santa Barbara Municipal Airport (KSBA)

3 flight lines (alternating E/W):
  Coastal_L01: 80.0 kilometer, az=90 degree
  Coastal_L02: 80.0 kilometer, az=270 degree
  Coastal_L03: 80.0 kilometer, az=90 degree


## 2. Still Air and Constant Wind

`StillAirField` always returns zero wind — it is equivalent to omitting
the `wind_source` parameter entirely. `ConstantWindField` wraps a single
speed and direction into U/V components using meteorological convention
(direction is where the wind blows *from*).

In [14]:
still = StillAirField()
u, v = still.wind_at(34.0, -119.0, 20000 * ureg.feet, datetime.datetime.now())
print(f"Still air:  U={u.m_as('knot'):.1f} kt, V={v.m_as('knot'):.1f} kt")

# 40 kt wind from the west (blows eastward)
westerly = ConstantWindField(wind_speed=40 * ureg.knot, wind_from_deg=270.0)
u, v = westerly.wind_at(34.0, -119.0, 20000 * ureg.feet, datetime.datetime.now())
print(f"40 kt from west:  U={u.m_as('knot'):+.1f} kt, V={v.m_as('knot'):+.1f} kt")

# 30 kt wind from the northwest
nw_wind = ConstantWindField(wind_speed=30 * ureg.knot, wind_from_deg=315.0)
u, v = nw_wind.wind_at(34.0, -119.0, 20000 * ureg.feet, datetime.datetime.now())
print(f"30 kt from NW:    U={u.m_as('knot'):+.1f} kt, V={v.m_as('knot'):+.1f} kt")

Still air:  U=0.0 kt, V=0.0 kt
40 kt from west:  U=+40.0 kt, V=+0.0 kt
30 kt from NW:    U=+21.2 kt, V=-21.2 kt


## 3. Comparing Flight Plans With and Without Wind

A 40 kt westerly wind is a headwind on eastbound legs and a tailwind
on westbound legs. Wind affects both the **timing** and the **geometry**
of Dubins transit paths (turning arcs become trochoids). Flight line
segments use a simple headwind/tailwind correction along the line
bearing, while transit, departure, and return legs use the full
wind-aware 3D Dubins solver.

In [15]:
plan_still = compute_flight_plan(
    aircraft=aircraft,
    flight_sequence=flight_lines,
    takeoff_airport=departure,
    return_airport=departure,
)

plan_wind = compute_flight_plan(
    aircraft=aircraft,
    flight_sequence=flight_lines,
    takeoff_airport=departure,
    return_airport=departure,
    wind_source=westerly,
)

# Side-by-side comparison
cols = ["segment_type", "segment_name", "time_to_segment"]
comparison = plan_still[cols].copy()
comparison.columns = ["type", "segment", "still_air_min"]
comparison["with_wind_min"] = plan_wind["time_to_segment"].values
comparison["delta_min"] = comparison["with_wind_min"] - comparison["still_air_min"]

print(comparison.to_string(index=False, float_format="%.2f"))
print(f"\nTotal still air:  {comparison['still_air_min'].sum():.1f} min")
print(f"Total with wind:  {comparison['with_wind_min'].sum():.1f} min")
print(f"Net difference:   {comparison['delta_min'].sum():+.1f} min")

       type                    segment  still_air_min  with_wind_min  delta_min
    takeoff                  Departure          10.06          10.74       0.68
flight_line                Coastal_L01          13.29          11.26      -2.03
    transit Coastal_L01 to Coastal_L02           2.13           2.13      -0.01
flight_line                Coastal_L02          13.29          16.21       2.92
    transit Coastal_L02 to Coastal_L03           2.13           2.59       0.46
flight_line                Coastal_L03          13.29          11.26      -2.03
    descent                     Return          13.39          13.40       0.01

Total still air:  67.6 min
Total with wind:  67.6 min
Net difference:   +0.0 min


## 4. MERRA-2 Reanalysis Winds (Historical)

For planning against historical data (“what were winds on this date
last year?”), use `wind_field_from_plan("merra2", ...)`. This fetches
3-hourly U/V winds on 42 pressure levels at 0.5° resolution via
OPeNDAP from NASA GES DISC.

**Prerequisites:** Set the `EARTHDATA_TOKEN` environment variable or
add your NASA Earthdata credentials to `~/.netrc`. Register at
https://urs.earthdata.nasa.gov if needed.

The `wind_field_from_plan` factory computes the geographic and temporal
bounding box from your flight sequence and fetches only the necessary
sub-region (server-side subsetting via OPeNDAP).

In [16]:
import os

# Only run this cell if Earthdata credentials are available
has_earthdata = (
    os.environ.get("EARTHDATA_TOKEN")
    or os.path.exists(os.path.expanduser("~/.netrc"))
)

if has_earthdata:
    takeoff_time = datetime.datetime(2024, 6, 15, 14, 0, tzinfo=datetime.timezone.utc)

    wf_merra2 = wind_field_from_plan(
        "merra2",
        flight_lines,
        takeoff_time,
        takeoff_airport=departure,
        return_airport=departure,
    )

    # Query the wind at a representative point
    u, v = wf_merra2.wind_at(
        34.4, -119.8, 20000 * ureg.feet, takeoff_time,
    )
    ws = np.sqrt(u.m_as("knot")**2 + v.m_as("knot")**2)
    wdir = (270 - np.degrees(np.arctan2(v.m_as("m/s"), u.m_as("m/s")))) % 360
    print(f"MERRA-2 wind at FL200, 2024-06-15 14:00 UTC:")
    print(f"  U = {u.m_as('knot'):+.1f} kt, V = {v.m_as('knot'):+.1f} kt")
    print(f"  Speed = {ws:.0f} kt from {wdir:.0f}\u00b0")
else:
    print("Earthdata credentials not found. Skipping MERRA-2 demo.")
    print("Set EARTHDATA_TOKEN or add credentials to ~/.netrc.")

MERRA-2 wind at FL200, 2024-06-15 14:00 UTC:
  U = +31.5 kt, V = +0.3 kt
  Speed = 32 kt from 269°


### Flight Plan with MERRA-2 Winds

Compare the flight plan computed with MERRA-2 reanalysis winds against
the still-air baseline.

In [17]:
if has_earthdata:
    plan_merra2 = compute_flight_plan(
        aircraft=aircraft,
        flight_sequence=flight_lines,
        takeoff_airport=departure,
        return_airport=departure,
        wind_source=wf_merra2,
        takeoff_time=takeoff_time,
    )

    # Compare
    cols = ["segment_type", "segment_name", "time_to_segment"]
    comp = plan_still[cols].copy()
    comp.columns = ["type", "segment", "still_air"]
    comp["merra2"] = plan_merra2["time_to_segment"].values
    comp["delta"] = comp["merra2"] - comp["still_air"]

    print(comp.to_string(index=False, float_format="%.2f"))
    print(f"\nTotal still air: {comp['still_air'].sum():.1f} min")
    print(f"Total MERRA-2:   {comp['merra2'].sum():.1f} min")
    print(f"Difference:      {comp['delta'].sum():+.1f} min")
else:
    print("Skipped (no Earthdata credentials).")

       type                    segment  still_air  merra2  delta
    takeoff                  Departure      10.06   10.33   0.27
flight_line                Coastal_L01      13.29   11.64  -1.65
    transit Coastal_L01 to Coastal_L02       2.13    1.88  -0.25
flight_line                Coastal_L02      13.29   15.43   2.14
    transit Coastal_L02 to Coastal_L03       2.13    2.48   0.35
flight_line                Coastal_L03      13.29   11.69  -1.60
    descent                     Return      13.39   13.39   0.00

Total still air: 67.6 min
Total MERRA-2:   66.8 min
Difference:      -0.7 min


## 5. GFS Forecast Winds (Operational)

NOAA's Global Forecast System (GFS) provides 0.25° resolution wind
forecasts out to 16 days, updated every 6 hours. No credentials
are needed.

`GFSWindField` uses the NOMADS GRIB filter for server-side subsetting —
only the requested variables, pressure levels, and geographic region
are downloaded (~10 KB vs ~500 MB for a full file).

`wind_field_from_plan` automatically selects the most recent GFS cycle
and the forecast hour closest to your mission time.

In [18]:
# Use a takeoff time a few hours from now (within GFS forecast range)
gfs_takeoff = datetime.datetime.now(tz=datetime.timezone.utc) + datetime.timedelta(hours=6)

try:
    wf_gfs = wind_field_from_plan(
        "gfs",
        flight_lines,
        gfs_takeoff,
        takeoff_airport=departure,
        return_airport=departure,
    )

    # Query the wind at a representative point
    u, v = wf_gfs.wind_at(34.4, -119.8, 20000 * ureg.feet, gfs_takeoff)
    ws = np.sqrt(u.m_as('knot')**2 + v.m_as('knot')**2)
    wdir = (270 - np.degrees(np.arctan2(v.m_as('m/s'), u.m_as('m/s')))) % 360
    print(f"GFS wind at FL200, {gfs_takeoff:%Y-%m-%d %H:%M} UTC:")
    print(f"  U = {u.m_as('knot'):+.1f} kt, V = {v.m_as('knot'):+.1f} kt")
    print(f"  Speed = {ws:.0f} kt from {wdir:.0f}\u00b0")
    has_gfs = True
except Exception as e:
    print(f"GFS unavailable: {e}")
    has_gfs = False

GFS wind at FL200, 2026-04-16 01:47 UTC:
  U = +22.2 kt, V = +10.4 kt
  Speed = 25 kt from 245°


### Flight Plan with GFS Winds

Compare the GFS-corrected flight plan against the still-air baseline.

In [19]:
if has_gfs:
    plan_gfs = compute_flight_plan(
        aircraft=aircraft,
        flight_sequence=flight_lines,
        takeoff_airport=departure,
        return_airport=departure,
        wind_source=wf_gfs,
        takeoff_time=gfs_takeoff,
    )

    cols = ["segment_type", "segment_name", "time_to_segment"]
    comp = plan_still[cols].copy()
    comp.columns = ["type", "segment", "still_air"]
    comp["gfs"] = plan_gfs["time_to_segment"].values
    comp["delta"] = comp["gfs"] - comp["still_air"]

    print(comp.to_string(index=False, float_format="%.2f"))
    print(f"\nTotal still air: {comp['still_air'].sum():.1f} min")
    print(f"Total GFS:       {comp['gfs'].sum():.1f} min")
    print(f"Difference:      {comp['delta'].sum():+.1f} min")
else:
    print("Skipped (GFS endpoint unavailable).")

       type                    segment  still_air   gfs  delta
    takeoff                  Departure      10.06 10.07   0.00
flight_line                Coastal_L01      13.29 12.09  -1.19
    transit Coastal_L01 to Coastal_L02       2.13  1.96  -0.17
flight_line                Coastal_L02      13.29 14.77   1.49
    transit Coastal_L02 to Coastal_L03       2.13  2.36   0.23
flight_line                Coastal_L03      13.29 12.12  -1.17
    descent                     Return      13.39 13.39   0.00

Total still air: 67.6 min
Total GFS:       66.8 min
Difference:      -0.8 min


## 6. GMAO GEOS-FP Winds (Near-Real-Time)

NASA GMAO provides GEOS-FP analysis fields at 0.25° resolution,
typically covering the last ~30 days. No credentials required.

> **Note:** The GMAO Fluid OPeNDAP server may be intermittently
> unavailable. This demo is wrapped in a try/except.


In [20]:
gmao_takeoff = datetime.datetime.now(tz=datetime.timezone.utc)

try:
    wf_gmao = wind_field_from_plan(
        "gmao",
        flight_lines,
        gmao_takeoff,
        takeoff_airport=departure,
        return_airport=departure,
    )

    u, v = wf_gmao.wind_at(34.4, -119.8, 20000 * ureg.feet, gmao_takeoff)
    ws = np.sqrt(u.m_as('knot')**2 + v.m_as('knot')**2)
    wdir = (270 - np.degrees(np.arctan2(v.m_as('m/s'), u.m_as('m/s')))) % 360
    print(f"GEOS-FP wind at FL200, {gmao_takeoff:%Y-%m-%d %H:%M} UTC:")
    print(f"  U = {u.m_as('knot'):+.1f} kt, V = {v.m_as('knot'):+.1f} kt")
    print(f"  Speed = {ws:.0f} kt from {wdir:.0f}°")
    has_gmao = True
except Exception as e:
    print(f"GMAO unavailable: {e}")
    has_gmao = False

GEOS-FP wind at FL200, 2026-04-15 19:47 UTC:
  U = +5.2 kt, V = -22.5 kt
  Speed = 23 kt from 347°


### Flight Plan with GEOS-FP Winds


In [21]:
if has_gmao:
    plan_gmao = compute_flight_plan(
        aircraft=aircraft,
        flight_sequence=flight_lines,
        takeoff_airport=departure,
        return_airport=departure,
        wind_source=wf_gmao,
        takeoff_time=gmao_takeoff,
    )

    cols = ["segment_type", "segment_name", "time_to_segment"]
    comp = plan_still[cols].copy()
    comp.columns = ["type", "segment", "still_air"]
    comp["gmao"] = plan_gmao["time_to_segment"].values
    comp["delta"] = comp["gmao"] - comp["still_air"]

    print(comp.to_string(index=False, float_format="%.2f"))
    print(f"\nTotal still air: {comp['still_air'].sum():.1f} min")
    print(f"Total GEOS-FP:   {comp['gmao'].sum():.1f} min")
    print(f"Difference:      {comp['delta'].sum():+.1f} min")
else:
    print("Skipped (GMAO endpoint unavailable).")

       type                    segment  still_air  gmao  delta
    takeoff                  Departure      10.06 10.06   0.00
flight_line                Coastal_L01      13.29 13.02  -0.26
    transit Coastal_L01 to Coastal_L02       2.13  2.07  -0.06
flight_line                Coastal_L02      13.29 13.73   0.44
    transit Coastal_L02 to Coastal_L03       2.13  2.19   0.06
flight_line                Coastal_L03      13.29 12.95  -0.34
    descent                     Return      13.39 13.39   0.00

Total still air: 67.6 min
Total GEOS-FP:   67.4 min
Difference:      -0.2 min


## 7. Using the Factory Function

The `wind_field_from_plan` factory is the recommended way to create wind
fields. It computes the spatial/temporal bounding box from your flight
sequence and only downloads the data you need.

| Source | Description |
|--------|-------------|
| `"still_air"` | Zero wind everywhere |
| `"merra2"` | NASA MERRA-2 reanalysis (historical) |
| `"gfs"` | NOAA GFS forecast (operational, 1-16 days) |
| `"gmao"` | NASA GEOS-FP near-real-time analysis |

In [22]:
# Still air via factory
wf_still = wind_field_from_plan(
    "still_air",
    flight_lines,
    takeoff_time=datetime.datetime.now(tz=datetime.timezone.utc),
)
u, v = wf_still.wind_at(34.0, -119.0, 20000 * ureg.feet, datetime.datetime.now())
print(f"still_air: U={u.m_as('knot'):.0f}, V={v.m_as('knot'):.0f} kt")

# Constant wind is created directly (not via factory)
wf_const = ConstantWindField(wind_speed=25 * ureg.knot, wind_from_deg=180.0)
u, v = wf_const.wind_at(34.0, -119.0, 20000 * ureg.feet, datetime.datetime.now())
print(f"constant 25 kt from S: U={u.m_as('knot'):+.1f}, V={v.m_as('knot'):+.1f} kt")

# GFS and MERRA-2 require network access:
# wf_gfs = wind_field_from_plan("gfs", flight_lines, takeoff_time)
# wf_merra2 = wind_field_from_plan("merra2", flight_lines, takeoff_time)

still_air: U=0, V=0 kt
constant 25 kt from S: U=-0.0, V=+25.0 kt
